# L1H3 — Primarily end of text token, diffuse across positions

**Layer 1, Head 3**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
    show_type_tokens, show_type_filtered_tables,
    get_type_positions, TYPE_ID_TO_POSITION_KEY,
    populate_measurable_type_heads,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)
populate_measurable_type_heads(cache, str_tokens)

## Types exhibited

In [ ]:
tm = compute_all_type_metrics(cache, str_tokens)
head_types = get_head_types(1, 3)
lines = []
for tid, act in head_types:
    pct_val = tm.get((tid, 1, 3))
    ent_key = TYPE_ENTROPY_KEYS.get(tid)
    ent_val = tm.get((ent_key, 1, 3)) if ent_key else None
    if pct_val is not None:
        ent_str = f", ent {ent_val:.1f}%" if ent_val is not None else ""
        lines.append(f"- **{HEAD_TYPES[tid][0]}** ({pct_val:.1f}%{ent_str})")
    else:
        lines.append(f"- **{HEAD_TYPES[tid][0]}** ({ACTIVITY_LABELS[act]})")
if not lines:
    lines.append("- *No types assigned yet (needs classification)*")
display(Markdown("\n".join(lines)))

## Attention Pattern Visualization

In [ ]:
show_head_pattern(str_tokens, cache, layer=1, head=3)

## Attention Weight Tables

Source to destination token pairs sorted by attention weight.

In [ ]:
attention = get_attention_pattern(cache, layer=1, head=3)
show_attention_tables(str_tokens, attention, top_k=25)

## Attention Metrics

In [ ]:
# Show all measurable metrics for this head (with entropy where available)
lines = []
for tid in HEAD_TYPES:
    key = (tid, 1, 3)
    if key in tm:
        ent_key = TYPE_ENTROPY_KEYS.get(tid)
        ent_val = tm.get((ent_key, 1, 3)) if ent_key else None
        ent_str = f"{ent_val:.2f}%" if ent_val is not None else "\u2014"
        lines.append(f"| {HEAD_TYPES[tid][0]} | {tm[key]:.2f}% | {ent_str} |")
# Show non-measurable assigned types
for tid, act in head_types:
    if (tid, 1, 3) not in tm:
        lines.append(f"| {HEAD_TYPES[tid][0]} | {ACTIVITY_LABELS[act]} | \u2014 |")
table = "| Type | Value | Entropy |\n|------|-------|---------|\n" + "\n".join(lines)
display(Markdown(table))